<a href="https://colab.research.google.com/github/Logan-Das/MMA3001_Project/blob/main/notebooks/02_grouped_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pork rasher QC: full pipeline (Google Colab)

Runs every experiment in the report in order. Set **Runtime → Change runtime type → T4 GPU** first.
Store your Roboflow key in Colab **Secrets** as `ROBOFLOW_API_KEY` (never paste it into the notebook).
All outputs go to Google Drive so a disconnect does not lose them.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/pork_rasher"

# Clone the repo (uses the GIT_TOKEN secret if the repo is private)
try:
    token = userdata.get("GIT_TOKEN")
    REPO = f"https://{token}@github.com/Logan-Das/MMA3001_Project.git"
except Exception:
    REPO = "https://github.com/Logan-Das/MMA3001_Project.git"
!git clone -q {REPO} /content/MMA3001_Project
%cd /content/MMA3001_Project
!pip install -q -r requirements.txt && pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Download the dataset WITHOUT augmentation
In Roboflow, create a new version with no augmentation (preprocessing only: auto-orient, resize 720×540) and put its number below.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
VERSION = 5   # <- the no-augmentation version number
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
ds = rf.workspace("hello-2aqe0").project("pork-rasher-error-packaging").version(VERSION).download("yolov11")
SRC = ds.location
print(SRC)

## 2. Leakage check and grouped split
Dry run first: read the gap percentiles and the number of groups the ORIGINAL split shares between splits.
Choose `--max-gap` where the gaps jump from "same tray" to "new tray", then build the split.

In [ ]:
!python scripts/make_grouped_split.py --src "$SRC" --max-gap 5 --dry-run

In [ ]:
MAX_GAP = 5   # <- set from the percentiles above, and justify it in the report
!python scripts/make_grouped_split.py --src "$SRC" --dst /content/pork_grouped --max-gap $MAX_GAP --seed 0
!cp /content/pork_grouped/split_manifest.csv $OUT/

### Independent check with a perceptual hash
Counts test images that have a near-identical image in train, for the original random split and the new grouped split. This does not use file names, so it validates the grouping rule.

In [ ]:
import numpy as np
from PIL import Image
from porkqc.grouping import dhash, cross_split_near_duplicates
from porkqc.labels import list_images

def hashes(folder):
    return np.array([dhash(np.asarray(Image.open(p).convert("L"))) for p in list_images(folder)])

for name, root, val in [("original", SRC, "valid"), ("grouped", "/content/pork_grouped", "val")]:
    tr, te = hashes(f"{root}/train/images"), hashes(f"{root}/test/images")
    dup = cross_split_near_duplicates(te, tr, max_distance=5)
    print(f"{name}: {dup.sum()} of {len(dup)} test images have a near-duplicate in train")

## 3. Train the baseline and the alternative with identical settings

In [ ]:
!python scripts/train.py --model yolo11n.pt --data /content/pork_grouped/data.yaml --out $OUT --name yolo11n_grouped

In [ ]:
!python scripts/train.py --model yolo11s.pt --data /content/pork_grouped/data.yaml --out $OUT --name yolo11s_grouped

## 4. Detection metrics per class (test split)

In [ ]:
from ultralytics import YOLO
for run in ["yolo11n_grouped", "yolo11s_grouped"]:
    m = YOLO(f"{OUT}/{run}/weights/best.pt").val(data="/content/pork_grouped/data.yaml", split="test",
                                                 project=OUT, name=f"{run}_test")

## 5. Tray-level pass/fail (threshold chosen on validation)

In [ ]:
for run in ["yolo11n_grouped", "yolo11s_grouped"]:
    !python scripts/evaluate_trays.py --weights $OUT/{run}/weights/best.pt --data /content/pork_grouped --out $OUT/results/{run}

## 6. Latency (warm-up, repeats, FP32 vs FP16)

In [ ]:
GFLOPS = {"yolo11n_grouped": 6.4, "yolo11s_grouped": 21.5}   # <- check the 'summary (fused)' line of each run
for run, g in GFLOPS.items():
    for half in ["", "--half"]:
        !python scripts/benchmark.py --weights $OUT/{run}/weights/best.pt --images /content/pork_grouped/test/images --gflops {g} {half} --out $OUT/results/{run}

## 7. Profile one prediction
Shows where time goes (pre-processing, model, NMS). Use this to explain the benchmark numbers.

In [ ]:
import cProfile, pstats
from ultralytics import YOLO
model = YOLO(f"{OUT}/yolo11n_grouped/weights/best.pt")
imgs = [str(p) for p in list_images("/content/pork_grouped/test/images")][:50]
model.predict(imgs[0], verbose=False)   # warm-up
cProfile.run("for p in imgs: model.predict(p, verbose=False)", "/content/pred.prof")
pstats.Stats("/content/pred.prof").sort_stats("cumulative").print_stats(15)